# 01 — Référentiel : législateurs & commissions

**Rôle :** construire la table d'identité (clé **BioGuideID**) avec parti, chambre, état/district, variantes de noms et appartenance aux commissions. C'est la pièce qui réconciliera les noms incohérents des PTR plus tard.

**Source :** `unitedstates/congress-legislators` (domaine public).

**Cellule 1 — Imports & chemins.** Ce notebook est autonome : il re-détecte la racine et les quelques dossiers dont il a besoin.

In [1]:
import requests, json
from collections import defaultdict
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
REFERENCE = PROJECT_ROOT / "data" / "reference"; REFERENCE.mkdir(parents=True, exist_ok=True)
RAW_REF   = REFERENCE / "raw"; RAW_REF.mkdir(parents=True, exist_ok=True)
REPORTS   = PROJECT_ROOT / "reports"; REPORTS.mkdir(parents=True, exist_ok=True)
USER_AGENT = "Ramify-QIS research (contact: <ton-email>)"
print("Référentiel ->", REFERENCE)

Référentiel -> /Users/lemairealice/Downloads/Jupiter/semaine 1/data/reference


**Cellule 2 — Petit téléchargeur JSON.** Réutilisable, avec cache disque (idempotent) et User-Agent honnête.

In [2]:
import yaml

BASE = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/"

def fetch_json(name: str, force: bool = False):
    # Télécharge {name}.yaml depuis congress-legislators (GitHub raw), avec cache disque.
    # Note: theunitedstates.io (ancienne URL JSON) retourne 403 — on passe au YAML source.
    path = RAW_REF / f"{name}.yaml"
    if path.exists() and not force:
        return yaml.safe_load(path.read_text())
    r = requests.get(BASE + f"{name}.yaml", headers={"User-Agent": USER_AGENT}, timeout=60)
    r.raise_for_status()
    data = yaml.safe_load(r.text)
    path.write_text(r.text)
    return data

**Cellule 3 — Téléchargement du référentiel.** Législateurs actuels + historiques, liste des commissions, et appartenance aux commissions.

In [3]:
legislators_current    = fetch_json("legislators-current")
legislators_historical = fetch_json("legislators-historical")
committees_current     = fetch_json("committees-current")
committee_membership   = fetch_json("committee-membership-current")

print("Actuels     :", len(legislators_current))
print("Historiques :", len(legislators_historical))
print("Commissions :", len(committees_current))

Actuels     : 537
Historiques : 12230
Commissions : 49


**Cellule 4 — Aplatissement d'un législateur.** On garde l'identité et le **dernier mandat connu** (parti, chambre, état, district).

In [4]:
def flatten_legislator(person: dict) -> dict:
    ids   = person.get("id", {})
    name  = person.get("name", {})
    terms = person.get("terms", [])
    last  = terms[-1] if terms else {}
    chamber = {"rep": "house", "sen": "senate"}.get(last.get("type"))
    return {
        "bioguide_id":   ids.get("bioguide"),
        "last_name":     name.get("last"),
        "first_name":    name.get("first"),
        "official_full": name.get("official_full"),
        "nickname":      name.get("nickname"),
        "party":         last.get("party"),
        "chamber":       chamber,
        "state":         last.get("state"),
        "district":      last.get("district"),
    }

rows = [flatten_legislator(p) for p in legislators_current + legislators_historical]
people = (pd.DataFrame(rows)
          .dropna(subset=["bioguide_id"])
          .drop_duplicates("bioguide_id")
          .reset_index(drop=True))
print("Législateurs uniques :", len(people))
people.head(3)

Législateurs uniques : 12767


,bioguide_id,last_name,first_name,official_full,nickname,party,chamber,state,district
0,C000127,Cantwell,Maria,Maria Cantwell,NaN,Democrat,senate,WA,NaN
1,K000367,Klobuchar,Amy,Amy Klobuchar,NaN,Democrat,senate,MN,NaN
2,S000033,Sanders,Bernard,Bernard Sanders,Bernie,Independent,senate,VT,NaN


**Cellule 5 — Variantes de nom.** Les PTR écrivent les noms de façon inconsistante (« Bill » vs « William »). On prépare toutes les formes connues + un nom canonique.

In [5]:
def name_variants(r) -> list:
    forms = set()
    if isinstance(r.official_full, str):
        forms.add(r.official_full)
    if isinstance(r.first_name, str) and isinstance(r.last_name, str):
        forms.add(f"{r.first_name} {r.last_name}")
    if isinstance(r.nickname, str) and isinstance(r.last_name, str):
        forms.add(f"{r.nickname} {r.last_name}")
    return sorted(forms)

people["declarant_name"] = people["official_full"].fillna(
    (people["first_name"].fillna("") + " " + people["last_name"].fillna("")).str.strip())
people["name_variants"] = people.apply(name_variants, axis=1)
people[["bioguide_id", "declarant_name", "name_variants"]].head(3)

,bioguide_id,declarant_name,name_variants
0,C000127,Maria Cantwell,[Maria Cantwell]
1,K000367,Amy Klobuchar,[Amy Klobuchar]
2,S000033,Bernard Sanders,"[Bernard Sanders, Bernie Sanders]"


**Cellule 6 — Nom des commissions.** On indexe les commissions par identifiant pour pouvoir les nommer.

In [6]:
committee_name = {c["thomas_id"]: c["name"] for c in committees_current}
list(committee_name.items())[:3]

[('HSAG', 'House Committee on Agriculture'),
 ('HSAP', 'House Committee on Appropriations'),
 ('HSAS', 'House Committee on Armed Services')]

**Cellule 7 — Membre → liste de commissions.** On rattache à chaque BioGuideID ses commissions.

In [7]:
member_committees = defaultdict(set)
for thomas_id, members in committee_membership.items():
    cname = committee_name.get(thomas_id, thomas_id)
    for m in members:
        if m.get("bioguide"):
            member_committees[m["bioguide"]].add(cname)

print("Membres avec >=1 commission :", len(member_committees))

Membres avec >=1 commission : 528


**Cellule 8 — Flag des commissions sensibles.** Finance/Banking, Armed Services (Defense) et Intelligence, par mots-clés (robuste aux variantes de noms).

> ⚠️ *Limite connue :* `committee-membership-current` ne couvre que le **Congrès actuel**. L'appartenance **historique** aux commissions (au moment d'un trade passé) est un point à compléter plus tard.

In [8]:
KEY_PATTERNS = ("financial", "banking", "finance", "armed services", "intelligence")

def committees_key_flag(committees: set) -> bool:
    blob = " | ".join(committees).lower()
    return any(p in blob for p in KEY_PATTERNS)

people["committee_membership"] = people["bioguide_id"].map(
    lambda b: sorted(member_committees.get(b, set())))
people["committees_key_flag"] = people["committee_membership"].map(committees_key_flag)
int(people["committees_key_flag"].sum())

197

**Cellule 9 — Sauvegarde du référentiel.** Les colonnes-listes sont sérialisées en JSON dans le CSV.

In [9]:
out = people.copy()
for col in ["name_variants", "committee_membership"]:
    out[col] = out[col].map(json.dumps)
ref_path = REFERENCE / "legislators.csv"
out.to_csv(ref_path, index=False)
print("Écrit :", ref_path, "-", len(out), "lignes")

Écrit : /Users/lemairealice/Downloads/Jupiter/semaine 1/data/reference/legislators.csv - 12767 lignes


**Cellule 10 — Mini-rapport de couverture.**

In [10]:
n = len(people)
n_comm = int((people["committee_membership"].map(len) > 0).sum())
n_key  = int(people["committees_key_flag"].sum())
print(f"Législateurs        : {n}")
print(f"Avec >=1 commission : {n_comm} ({n_comm/n:.0%})  [Congrès actuel]")
print(f"Sur commission clé  : {n_key}")

Législateurs        : 12767
Avec >=1 commission : 528 (4%)  [Congrès actuel]
Sur commission clé  : 197


Référentiel prêt ✅ — passez à **`02_house_index`**.